In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import shap
from imblearn.over_sampling import SMOTE


/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from collections import Counter

# Load data (handling potential byte order mark)
admission_annotation = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionGenes/Raw Data Files/admission_annotation.csv')
gene_symbols = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionGenes/Raw Data Files/gene_symbols.csv')
admission_norm_gene_exp_df = pd.read_csv('/Users/garimau/dev/dsdp-ucsf/UCSFDelirium/AdmissionGenes/Raw Data Files/admission_norm_gene_exp_df.csv', index_col=0)


# Merge and clean gene expression data
admission_norm_gene_exp_df = admission_norm_gene_exp_df.merge(gene_symbols, left_index=True, right_on='gene_ids', how='left')
admission_norm_gene_exp_df = admission_norm_gene_exp_df.set_index('gene_symbols').drop(['gene_ids', 'Unnamed: 0'], axis=1)
admission_norm_gene_exp_df = admission_norm_gene_exp_df.dropna(axis=0)

# Remove hyphens from 'X' column in admission_annotation
admission_annotation['X'] = admission_annotation['X'].str.replace('-', '')

# Transpose, merge, and prepare data
X = admission_norm_gene_exp_df.transpose()
X = X.merge(admission_annotation, left_index=True, right_on='X', how='inner').set_index('X')
X.to_csv('admission.csv', index=False)
X.head()

,RN7SL2,HBB,HBA2,HBA1,RNU4-2,ALAS2,NaN,CA1,HBD,RN7SL4P,...,RBM8A,ANKRD10,C6orf89,ARIH2,UBE2I,CNOT10,UBE2K,TBRG1,Diagnosis,Steroids
X,,,,,,,,,,,,,,,,,,,,,
MVIR1HS101D0PBMC1RSQ1,18.685163,17.932175,16.035154,13.760147,13.842371,12.345106,13.822952,9.934184,8.572786,13.459054,...,10.928632,11.158368,11.004793,11.349125,11.646860,9.814219,10.621316,10.634878,1,0
MVIR1HS107D0PBMC1RSQ1,17.644940,20.928008,18.412823,16.476078,13.123230,14.270686,12.871229,12.539489,13.922281,12.290336,...,10.563948,11.136085,10.780020,11.238639,11.471842,9.635421,10.415488,10.494257,1,0
MVIR1HS109D0PBMC1RSQ1,17.109929,19.025080,15.875361,15.005524,12.992272,13.143869,12.205144,11.160206,10.579923,11.665192,...,10.730673,11.018725,10.744570,11.125231,11.680329,9.608055,10.407175,10.517881,0,0
MVIR1HS10D0PBMC1RSQ4,8.317684,19.268316,16.648941,15.397664,6.048804,11.103061,5.234901,10.042230,9.044351,5.902330,...,10.500445,11.315396,10.983885,10.810117,11.348690,9.811048,10.761488,11.001319,0,0
MVIR1HS111D0PBMC1RSQ1,17.411885,11.947249,9.720248,7.927620,13.171352,6.463404,12.816903,6.025628,6.094624,12.210416,...,10.583444,11.153840,10.584126,10.944706,11.592284,9.722730,10.431077,10.453642,0,0


In [4]:
y = X['Diagnosis']
X = X.drop('Diagnosis', axis=1)
X.columns = X.columns.astype(str)

In [5]:
oversample = SMOTE()
counter = Counter(y)
print(counter)
X.head()

X, y = oversample.fit_resample(X, y)
# summarize the new class distribution
# scatter plot of examples by class label
X.shapecounter = Counter(y)
print(counter)

# Assuming X is a DataFrame and y is a Series or array
X_resampled = pd.DataFrame(X)
y_resampled = pd.Series(y, name='target')

# Concatenate X and y
df_resampled = pd.concat([X_resampled, y_resampled], axis=1)

# Save to CSV
df_resampled.to_csv('oversampled_admission.csv', index=False)


Counter({0: 88, 1: 24})


/var/folders/ql/sqk652rs4vdbmts6q6jhtcv00000gn/T/ipykernel_3935/1948636158.py:9: UserWarning: Pandas doesn't allow columns to be created via a new attribute name - see https://pandas.pydata.org/pandas-docs/stable/indexing.html#attribute-access
  X.shapecounter = Counter(y)


Counter({0: 88, 1: 24})
